# Hybrid fine-tune GPT-2 

Notebook này giữ backbone `NlpHUST/gpt2-vietnamese`, chạy Kaggle Internet OFF, và tối ưu đáp án số cuối thay vì lời giải dài.

Pipeline:
- Load `train.json` mới: `/kaggle/input/datasets/phamanhtuanas/gpt2-math/keep_asy_if_short/train.json`.
- `valid.json` chỉ load tham khảo vì lỗi; validation được sinh từ train bằng grouped stratified split.
- Gold chỉ lấy từ `response_vi`/`response`; input chỉ lấy `query_vi`/`query`.
- So sánh ablation: baseline, LoRA target gốc, answer-only, type prompt, voting, full pipeline.
- Full pipeline: rule solver deterministic -> TF-IDF retrieval -> LoRA generation + majority voting.
- Xuất `valid_output.json`, `valid_report.json`, `test_predictions.json`.

In [1]:
# 1. Imports, paths, config
import os, sys, re, ast, gc, json, math, time, random, inspect, hashlib
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.neighbors import NearestNeighbors
    SKLEARN_OK = True
except Exception as exc:
    SKLEARN_OK = False
    print("WARNING sklearn unavailable:", repr(exc))

try:
    from peft import LoraConfig, PeftModel, get_peft_model
    PEFT_OK = True
except Exception as exc:
    PEFT_OK = False
    print("WARNING peft unavailable; fallback partial fine-tune:", repr(exc))

try:
    from IPython.display import display
except Exception:
    display = print

try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "hybrid_finetune"
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = WORK_DIR / "hybrid_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

SAFE_EOS_ID = 50256
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def first_existing(*paths):
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào:\n" + "\n".join(checked))

def first_existing_optional(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

TRAIN_FILE = first_existing(
    "/kaggle/input/datasets/phamanhtuanas/gpt2-math/keep_asy_if_short/train.json",
    "/kaggle/input/gpt2-math/keep_asy_if_short/train.json",
    "/kaggle/input/gpt2-math/train.json",
    PROJECT_ROOT / "data" / "train.json",
)
DATA_DIR = TRAIN_FILE.parent
VALID_FILE_BUGGY = first_existing_optional(DATA_DIR / "valid.json", PROJECT_ROOT / "data" / "valid.json")
TEST_FILE = first_existing_optional(DATA_DIR / "test.json", "/kaggle/input/test.json", PROJECT_ROOT / "data" / "test.json")
MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese",
    PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
)

VALID_OUTPUT_PATH = WORK_DIR / "valid_output.json"
VALID_REPORT_PATH = WORK_DIR / "valid_report.json"
TEST_OUTPUT_PATH = WORK_DIR / "test_predictions.json"
FINAL_REPORT_PATH = WORK_DIR / "final_report.md"

CUDA_OK = torch.cuda.is_available()
LOCAL_DRY_RUN = not IS_KAGGLE
ALLOW_CPU_TRAINING = False
# Đặt RUN_FINAL_ONLY=True sau khi đã chọn config tốt nhất để bỏ qua ablation và tiết kiệm thời gian official run.
RUN_FINAL_ONLY = False
FINAL_TARGET_MODE = "auto"  # "auto", "answer_only", hoặc "calc_tail"
RUN_QUICK_ABLATIONS = (not LOCAL_DRY_RUN) and (not RUN_FINAL_ONLY)
RUN_FINAL_TRAIN = not LOCAL_DRY_RUN
RUN_VALIDATION = True
RUN_TEST_INFERENCE = True

VALID_FRACTION = 0.08
VALID_PER_TYPE = 350
MAX_TRAIN_SAMPLES = None
TRAIN_SAMPLE_PER_TYPE_FOR_ABLATION = 700
VALID_SAMPLE_PER_TYPE_FOR_ABLATION = 120
TRAINER_EVAL_SAMPLES = 1200
MAX_LENGTH_ORIGINAL = 512
MAX_LENGTH_SHORT = 384

LORA_CONFIG = dict(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["c_attn", "c_proj", "c_fc"], fan_in_fan_out=True,
)
FINAL_TRAINING = dict(
    num_train_epochs=4, learning_rate=2e-4, per_device_train_batch_size=8,
    per_device_eval_batch_size=8, gradient_accumulation_steps=2,
    warmup_ratio=0.05, weight_decay=0.01, logging_steps=50, max_steps=None,
)
ABLATION_TRAINING = dict(
    num_train_epochs=1, learning_rate=2e-4, per_device_train_batch_size=8,
    per_device_eval_batch_size=8, gradient_accumulation_steps=2,
    warmup_ratio=0.05, weight_decay=0.01, logging_steps=25, max_steps=280,
)
GENERATION = dict(num_return_sequences=8, temperature=0.7, top_p=0.95, max_new_tokens=48, fallback_max_new_tokens=32, batch_size=8)
RULE_HIGH_CONFIDENCE = 0.95
RULE_REASON_ALLOWLIST = {
    "explicit_arithmetic",
    "max_lcm_pairs",
    "lcm",
    "gcd",
    "arithmetic_sequence_count",
    "isosceles_triangle_perimeter",
    "rectangle_perimeter",
    "square_perimeter",
    "rectangle_area",
    "square_area",
    "triangle_area",
    "cube_volume",
    "box_volume",
    "race_average_time_per_km",
    "dozen",
    "hour_to_minute",
}
RETRIEVAL_DEFAULT_THRESHOLD = 0.92
RETRIEVAL_FEWSHOT_THRESHOLD = 0.90
RETRIEVAL_TOP_K = 5
RETRIEVAL_CLOSE_DELTA = 0.01

print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__, "| CUDA:", CUDA_OK, "| PEFT:", PEFT_OK, "| sklearn:", SKLEARN_OK)
if CUDA_OK:
    print("GPU:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
print("TRAIN_FILE:", TRAIN_FILE)
print("VALID_FILE_BUGGY:", VALID_FILE_BUGGY)
print("TEST_FILE:", TEST_FILE)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_DIR:", WORK_DIR)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128 | CUDA: True | PEFT: True | sklearn: True
GPU: Tesla T4 (7, 5)
TRAIN_FILE: /kaggle/input/datasets/phamanhtuanas/gpt2-math/keep_asy_if_short/train.json
VALID_FILE_BUGGY: None
TEST_FILE: None
MODEL_DIR: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
WORK_DIR: /kaggle/working


In [2]:
# 2. Load JSON/JSONL, normalize schema query/response -> query_vi/response_vi
TYPE_ORDER = ["GSM_Rephrased","GSM_AnsAug","GSM_SV","GSM_FOBAR","MATH_Rephrased","MATH_AnsAug","MATH_SV","MATH_FOBAR"]

def save_json(obj, path, indent=2):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)

def load_json_or_jsonl(path):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        while True:
            ch = f.read(1)
            if ch == "":
                return []
            if not ch.isspace():
                break
        f.seek(0)
        if ch == "[":
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError(f"{path} must be JSON array or JSONL")
            return data
        out = []
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Bad JSONL at {path}:{line_no}: {exc}") from exc
        return out

def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "").replace("\ufeff", "").replace("\u200b", "")).strip()

def stable_hash(text):
    return hashlib.blake2b(normalize_space(text).lower().encode("utf-8"), digest_size=16).hexdigest()

def pick(rec, names, default=""):
    for name in names:
        val = rec.get(name)
        if val is not None and str(val).strip():
            return val
    return default

def normalize_record(rec, idx, split):
    q = pick(rec, ["query_vi", "query", "question_vi", "problem", "input", "original_question_vi"])
    r = pick(rec, ["response_vi", "response", "answer_vi", "solution", "target"], "")
    typ = pick(rec, ["type", "problem_type", "category"], "unknown")
    out = dict(rec)
    out.update(id=rec.get("id", f"{split}_{idx:06d}"), query_vi=str(q).strip(), response_vi=str(r).strip(), type=str(typ).strip() or "unknown", _row_index=idx, _source_split=split)
    out["_group_key"] = str(rec.get("original_question_hash") or rec.get("query_hash") or rec.get("query_response_hash") or stable_hash(out["query_vi"]))
    return out

def normalize_records(records, split, require_response):
    out, dropped = [], Counter()
    for idx, rec in enumerate(records):
        item = normalize_record(rec, idx, split)
        if not item["query_vi"]:
            dropped["empty_query"] += 1; continue
        if require_response and not item["response_vi"]:
            dropped["empty_response"] += 1; continue
        out.append(item)
    print(f"{split}: kept={len(out)} dropped={dict(dropped)}")
    return out

raw_train_records = normalize_records(load_json_or_jsonl(TRAIN_FILE), "train", True)
raw_buggy_valid_records = []
if VALID_FILE_BUGGY:
    try:
        raw_buggy_valid_records = normalize_records(load_json_or_jsonl(VALID_FILE_BUGGY), "valid_buggy_unused", True)
        print("valid.json loaded only for schema reference; not used as main validation.")
    except Exception as exc:
        print("Skip buggy valid.json:", repr(exc))
raw_test_records = normalize_records(load_json_or_jsonl(TEST_FILE), "test", False) if TEST_FILE else []

print("Train type counts:")
display(pd.Series(Counter(r["type"] for r in raw_train_records)).sort_index().to_frame("count"))
print(json.dumps(raw_train_records[0], ensure_ascii=False, indent=2)[:1500])

train: kept=88907 dropped={}
Train type counts:


,count
GSM_AnsAug,17866
GSM_FOBAR,9149
GSM_Rephrased,18162
GSM_SV,8902
MATH_AnsAug,16653
MATH_FOBAR,3376
MATH_Rephrased,11436
MATH_SV,3363


{
  "id": "train_000000",
  "raw_index": 0,
  "dataset_name": "train",
  "split": "train",
  "query": "Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?",
  "response": "Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó, tức là 84 * 2/3 = 56 khách. Vậy tổng số khách là 84 + 56 = 140 khách. Người phục vụ luôn làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây\nĐáp án là: 1200",
  "final_answer": "1200",
  "answer_conflict_key": "1200",
  "type": "GSM_AnsAug",
  "source_group": "GSM",
  "aug_type": "AnsAug"

In [3]:
# 3. Robust final-answer extraction và evaluator local
ANCHOR_RE = re.compile(r"(?:Đáp\s*án\s*(?:là)?\s*[:：]?|Câu\s*trả\s*lời\s*(?:là)?\s*[:：]?|####\s*|Final\s*answer\s*[:：]?|Answer\s*[:：]?)", re.I)
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]+)\}")
LATEX_FRAC_RE = re.compile(r"\\(?:d?frac)\s*\{([^{}]+)\}\s*\{([^{}]+)\}")
SIMPLE_FRAC_RE = re.compile(r"(?<![\w])[-+]?\d+(?:[\.,]\d+)?\s*/\s*[-+]?\d+(?:[\.,]\d+)?(?![\w])")
NUMBER_RE = re.compile(r"[-+]?\d+(?:[\.,]\d+)*(?:%?)")
PI_EXPR_RE = re.compile(r"[-+]?\s*(?:(?:\d+(?:[\.,]\d+)?)\s*\*?\s*)?(?:\\pi|π|pi)(?:\s*/\s*[-+]?\d+(?:[\.,]\d+)?)?", re.I)

def strip_latex_noise(text):
    return str(text or "").replace("$", " ").replace("\\left", "").replace("\\right", "").replace("\\,", " ").replace("−", "-").replace("–", "-").replace("—", "-")

def normalize_numeric_token(tok):
    tok = str(tok or "").strip().replace("%", "").replace(" ", "")
    if not tok: return tok
    sign = ""
    if tok[0] in "+-":
        sign, tok = tok[0], tok[1:]
    if "," in tok and "." in tok:
        dec = "," if tok.rfind(",") > tok.rfind(".") else "."
        thou = "." if dec == "," else ","
        tok = tok.replace(thou, "").replace(dec, ".")
    elif "," in tok or "." in tok:
        sep = "," if "," in tok else "."
        parts = tok.split(sep)
        if len(parts) > 2 and all(len(p) == 3 for p in parts[1:]):
            tok = "".join(parts)
        elif len(parts) == 2:
            left, right = parts
            # 10,100 và 10.100 thường là hàng nghìn; 1,5 và 0,50 là thập phân.
            tok = left + right if len(right) == 3 and len(left) <= 3 and left not in {"0", ""} else left + "." + right
        else:
            tok = tok.replace(sep, "")
    return sign + tok

def parse_decimal(tok):
    tok = normalize_numeric_token(tok)
    if not tok or tok in {"+", "-", "."}: return None
    try:
        return float(Decimal(tok))
    except (InvalidOperation, ValueError):
        try: return float(tok)
        except ValueError: return None

def safe_eval_numeric_expr(expr):
    expr = strip_latex_noise(expr).lower().replace("\\pi", "pi").replace("π", "pi").replace("^", "**")
    expr = re.sub(r"(?<=\d)\s*pi", "*pi", expr)
    expr = re.sub(r"pi\s*(?=\d)", "pi*", expr)
    allowed = {"pi": math.pi}
    nodes = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd, ast.Constant, ast.Load, ast.Name)
    try: tree = ast.parse(expr, mode="eval")
    except SyntaxError: return None
    for node in ast.walk(tree):
        if not isinstance(node, nodes): return None
        if isinstance(node, ast.Name) and node.id not in allowed: return None
        if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)): return None
    try: val = eval(compile(tree, "<expr>", "eval"), {"__builtins__": {}}, allowed)
    except Exception: return None
    return float(val) if isinstance(val, (int, float)) and math.isfinite(float(val)) else None

def clean_candidate(text):
    text = strip_latex_noise(text)
    text = re.sub(r"[\]\[(){}]", " ", text)
    text = re.sub(r"(?:đồng|usd|cm|kg|m|km|phút|giờ|ngày|đô|quả|cái|chiếc|người)\b", " ", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip(" .,:;\n\t")

def parse_answer_number(candidate):
    if candidate is None: return None
    raw = strip_latex_noise(candidate)
    # Parse structured forms before removing braces in clean_candidate().
    m = LATEX_FRAC_RE.search(raw)
    if m:
        a, b = parse_answer_number(m.group(1)), parse_answer_number(m.group(2))
        return a / b if a is not None and b not in (None, 0) else None
    text = clean_candidate(raw)
    if not text: return None
    m = SIMPLE_FRAC_RE.search(text)
    if m:
        a, b = re.split(r"/", m.group(0), maxsplit=1)
        a, b = parse_decimal(a), parse_decimal(b)
        return a / b if a is not None and b not in (None, 0) else None
    m = PI_EXPR_RE.search(text)
    if m:
        val = safe_eval_numeric_expr(m.group(0))
        if val is not None: return val
    if len(text) <= 80 and re.search(r"[+\-*/^()]", text) and re.fullmatch(r"[\d\s\.,+\-*/^()\\piπpi]+", text, flags=re.I):
        expr = re.sub(r"\d+(?:[\.,]\d+)*", lambda x: normalize_numeric_token(x.group(0)), text)
        val = safe_eval_numeric_expr(expr)
        if val is not None: return val
    nums = NUMBER_RE.findall(text)
    return parse_decimal(nums[0]) if nums else None

def find_answer_candidates(text):
    text = strip_latex_noise(text)
    cand, spans = [], []
    for rx in [LATEX_FRAC_RE, SIMPLE_FRAC_RE, PI_EXPR_RE]:
        for m in rx.finditer(text):
            cand.append((m.start(), -(m.end() - m.start()), m.group(0)))
            spans.append((m.start(), m.end()))
    for m in NUMBER_RE.finditer(text):
        if any(a <= m.start() < b for a, b in spans):
            continue
        cand.append((m.start(), -(m.end() - m.start()), m.group(0)))
    return [x for _, _, x in sorted(cand)]

def canonical_answer_text(value, original=None):
    if value is None: return None
    if abs(value - round(value)) <= 1e-9: return str(int(round(value)))
    if original is not None and "/" in str(original) and "frac" not in str(original):
        return re.sub(r"\s+", "", str(original).strip())
    s = f"{float(value):.12g}"
    return s.rstrip("0").rstrip(".") if "." in s else s

def extract_final_answer(text):
    if text is None: return None
    text = str(text)
    chunks = []
    for m in ANCHOR_RE.finditer(text):
        tail = re.split(r"(?:\n\s*\n|\n\s*(?:Bài toán|Câu hỏi|Question)\s*:)", text[m.end():], maxsplit=1)[0]
        chunks.append(tail[:240])
    chunks.extend(BOXED_RE.findall(text))
    chunks.append(text[-500:])
    for chunk in list(reversed(chunks[:-1])) + [chunks[-1]]:
        cands = find_answer_candidates(chunk)
        if not cands: continue
        for cand in (cands if chunk != chunks[-1] else list(reversed(cands))):
            val = parse_answer_number(cand)
            if val is not None: return canonical_answer_text(val, cand)
    return None

def relative_error(pred, gold):
    if pred is None or gold is None: return None
    if not (math.isfinite(float(pred)) and math.isfinite(float(gold))): return None
    return abs(float(pred) - float(gold)) / max(1.0, abs(float(gold)))

def score_from_rel(rel):
    if rel is None: return 0
    if rel <= 0.01: return 10
    if rel <= 0.10: return 5
    if rel <= 0.50: return 1
    return 0

def evaluate_answer_texts(pred_texts, gold_texts, types=None, ids=None):
    rows = []
    for i, (pred_text, gold_text) in enumerate(zip(pred_texts, gold_texts)):
        pa, ga = extract_final_answer(pred_text), extract_final_answer(gold_text)
        pv, gv = parse_answer_number(pa), parse_answer_number(ga)
        rel = relative_error(pv, gv); score = score_from_rel(rel)
        rows.append(dict(id=ids[i] if ids else i, type=types[i] if types else "unknown", pred_answer=pa, gold_answer=ga, pred_value=pv, gold_value=gv, relative_error=rel, score=score, exact=pa is not None and ga is not None and pa == ga, pass_1pct=rel is not None and rel <= 0.01, pass_10pct=rel is not None and rel <= 0.10, pass_50pct=rel is not None and rel <= 0.50, not_extractable=pa is None))
    return rows

def summarize_eval_rows_no_type(rows):
    n = len(rows); raw = sum(r["score"] for r in rows)
    return dict(num_samples=n, raw_score=raw, score_10=raw/max(1,n), score_pct_of_max=raw/max(1,10*n), exact_rate=float(np.mean([r["exact"] for r in rows])) if rows else 0.0, pass_1pct_rate=float(np.mean([r["pass_1pct"] for r in rows])) if rows else 0.0, pass_10pct_rate=float(np.mean([r["pass_10pct"] for r in rows])) if rows else 0.0, pass_50pct_rate=float(np.mean([r["pass_50pct"] for r in rows])) if rows else 0.0, not_extractable_count=int(sum(r["not_extractable"] for r in rows)))

def summarize_eval_rows(rows):
    out = summarize_eval_rows_no_type(rows)
    out["max_raw_score"] = 10 * len(rows)
    out["by_type"] = {t: summarize_eval_rows_no_type([r for r in rows if r["type"] == t]) for t in sorted(set(r["type"] for r in rows))}
    return out

for s in ["Đáp án là: 10,100", "Câu trả lời là: -3.5", r"#### \frac{3}{4}", r"Kết quả là $2\pi$.", "Vậy x=1/8. Đáp án là: 1/8"]:
    print(s, "=>", extract_final_answer(s), parse_answer_number(extract_final_answer(s)))

Đáp án là: 10,100 => 10100 10100.0
Câu trả lời là: -3.5 => -3.5 -3.5
#### \frac{3}{4} => 0.75 0.75
Kết quả là $2\pi$. => 6.28318530718 6.28318530718
Vậy x=1/8. Đáp án là: 1/8 => 1/8 0.125


In [4]:
# 4. Extract gold, split validation từ train
for rec in raw_train_records:
    rec["gold_answer"] = extract_final_answer(rec["response_vi"])
    rec["gold_value"] = parse_answer_number(rec["gold_answer"])

extract_fail = [r for r in raw_train_records if r["gold_answer"] is None]
usable_records = [r for r in raw_train_records if r["gold_answer"] is not None]
print("Gold answer extract failed:", len(extract_fail), "/", len(raw_train_records))

def make_grouped_valid_split(records, valid_fraction=0.08, valid_per_type=350, seed=42):
    rng = random.Random(seed)
    group_to_records = defaultdict(list)
    for rec in records:
        group_to_records[rec["_group_key"]].append(rec)
    type_to_groups = defaultdict(list)
    for g, items in group_to_records.items():
        typ = Counter(x["type"] for x in items).most_common(1)[0][0]
        type_to_groups[typ].append(g)
    counts = Counter(r["type"] for r in records)
    target = {t: min(valid_per_type, max(1, int(round(c * valid_fraction)))) for t, c in counts.items()}
    valid_groups, valid_counts = set(), Counter()
    for typ, groups in sorted(type_to_groups.items()):
        groups = list(groups); rng.shuffle(groups)
        for g in groups:
            if valid_counts[typ] >= target[typ]: break
            if g in valid_groups: continue
            valid_groups.add(g)
            for item in group_to_records[g]:
                valid_counts[item["type"]] += 1
    train, valid = [], []
    for rec in records:
        (valid if rec["_group_key"] in valid_groups else train).append(rec)
    return train, valid, dict(target_by_type=target, actual_by_type=dict(valid_counts), valid_groups=len(valid_groups))

train_records_all, valid_records, split_report = make_grouped_valid_split(usable_records, VALID_FRACTION, VALID_PER_TYPE, SEED)
if MAX_TRAIN_SAMPLES is not None and len(train_records_all) > MAX_TRAIN_SAMPLES:
    train_records = []
    rng = random.Random(SEED)
    by_type = defaultdict(list)
    for r in train_records_all: by_type[r["type"]].append(r)
    per_type = max(1, MAX_TRAIN_SAMPLES // max(1, len(by_type)))
    for items in by_type.values():
        rng.shuffle(items); train_records.extend(items[:per_type])
    rng.shuffle(train_records); train_records = train_records[:MAX_TRAIN_SAMPLES]
else:
    train_records = train_records_all
random.Random(SEED).shuffle(train_records)
random.Random(SEED).shuffle(valid_records)

print("Split report:", json.dumps(split_report, ensure_ascii=False, indent=2))
print("Train:", len(train_records), "| Valid:", len(valid_records))
display(pd.DataFrame({"train": pd.Series(Counter(r["type"] for r in train_records)), "valid": pd.Series(Counter(r["type"] for r in valid_records))}).fillna(0).astype(int))

gold_rows = evaluate_answer_texts([f"Đáp án là: {r['gold_answer']}" for r in valid_records], [r["response_vi"] for r in valid_records], [r["type"] for r in valid_records], [r["id"] for r in valid_records])
print("Gold self-eval score_10:", summarize_eval_rows(gold_rows)["score_10"])

Gold answer extract failed: 20 / 88907
Split report: {
  "target_by_type": {
    "GSM_AnsAug": 350,
    "MATH_AnsAug": 350,
    "GSM_SV": 350,
    "GSM_FOBAR": 350,
    "MATH_Rephrased": 350,
    "MATH_SV": 269,
    "GSM_Rephrased": 350,
    "MATH_FOBAR": 270
  },
  "actual_by_type": {
    "GSM_AnsAug": 596,
    "GSM_SV": 353,
    "GSM_Rephrased": 416,
    "GSM_FOBAR": 423,
    "MATH_AnsAug": 559,
    "MATH_Rephrased": 384,
    "MATH_SV": 274,
    "MATH_FOBAR": 339
  },
  "valid_groups": 488
}
Train: 85543 | Valid: 3344


,train,valid
GSM_AnsAug,17270,596
GSM_FOBAR,8726,423
GSM_Rephrased,17746,416
GSM_SV,8549,353
MATH_AnsAug,16078,559
MATH_FOBAR,3037,339
MATH_Rephrased,11048,384
MATH_SV,3089,274


Gold self-eval score_10: 10.0


In [5]:
# 5. Prompt theo type, target modes, tokenizer/dataset
TYPE_INSTRUCTION_MAP = {
    "GSM_Rephrased": "Giải bài toán số học đời sống và trả về đáp án cuối.",
    "GSM_AnsAug": "Giải bài toán số học đời sống và trả về đáp án cuối.",
    "MATH_Rephrased": "Giải bài toán toán học và trả về đáp án cuối.",
    "MATH_AnsAug": "Giải bài toán toán học và trả về đáp án cuối.",
    "GSM_SV": "Đây là bài tìm biến chưa biết. Câu hỏi cuối cùng mới là mục tiêu. Chỉ trả về giá trị biến.",
    "MATH_SV": "Đây là bài tìm biến chưa biết. Câu hỏi cuối cùng mới là mục tiêu. Chỉ trả về giá trị biến.",
    "GSM_FOBAR": "Đây là bài đảo ngược từ đáp án đã biết để tìm biến x. Không trả lại đáp án bài gốc.",
    "MATH_FOBAR": "Đây là bài đảo ngược từ đáp án đã biết để tìm biến x. Không trả lại đáp án bài gốc.",
}
GENERIC_INSTRUCTION = "Giải bài toán và trả về đáp án cuối."

def type_instruction(typ, type_specific=True):
    return TYPE_INSTRUCTION_MAP.get(str(typ), GENERIC_INSTRUCTION) if type_specific else GENERIC_INSTRUCTION

def build_answer_prompt(rec, type_specific=True, fewshot=None):
    parts = []
    if fewshot:
        parts.append(f"Ví dụ gần giống:\nBài toán: {fewshot['query_vi'].strip()}\nĐáp án: {fewshot['gold_answer']}")
    parts.append(f"Nhiệm vụ: {type_instruction(rec.get('type'), type_specific)}\nBài toán: {rec['query_vi'].strip()}\nĐáp án:")
    return "\n\n".join(parts)

def build_solution_prompt(rec, type_specific=True):
    return f"Nhiệm vụ: {type_instruction(rec.get('type'), type_specific)}\nBài toán: {rec['query_vi'].strip()}\nLời giải:\n"

def make_target_text(rec, target_mode):
    if target_mode == "calc_tail":
        body = ANCHOR_RE.split(rec["response_vi"])[0]
        parts = re.split(r"(?<=[.!?])\s+|\n+", body)
        parts = [s for s in parts if re.search(r"\d|=", s)]
        tail = " ".join(parts[-4:])[:450]
        if not tail:
            tail = "Tính theo dữ kiện trong đề."
        return f"{tail}\nĐáp án là: {rec['gold_answer']}"
    if target_mode == "original": return rec["response_vi"].strip()
    if target_mode == "answer_only": return " " + str(rec["gold_answer"]).strip()
    if target_mode == "short_solution": return f"Lời giải: Tính theo dữ kiện trong đề.\nĐáp án là: {rec['gold_answer']}"
    raise ValueError(target_mode)

def build_training_pair(rec, target_mode="answer_only", type_specific=True):
    prompt = build_answer_prompt(rec, type_specific) if target_mode == "answer_only" else build_solution_prompt(rec, type_specific)
    return prompt, make_target_text(rec, target_mode)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
if getattr(tokenizer, "pad_token", None) is None and getattr(tokenizer, "eos_token", None) is not None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer vocab_size:", getattr(tokenizer, "vocab_size", None), "| len:", len(tokenizer), "| pad/eos:", tokenizer.pad_token_id, tokenizer.eos_token_id)

def fit_prompt_target(prompt_ids, target_ids, max_length):
    if len(prompt_ids) + len(target_ids) <= max_length:
        return prompt_ids, target_ids
    target_budget = min(len(target_ids), max(16, max_length // 3))
    if len(target_ids) > target_budget:
        target_ids = target_ids[-target_budget:]
    prompt_budget = max(1, max_length - len(target_ids))
    return prompt_ids[-prompt_budget:], target_ids

class MathSFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length, target_mode="answer_only", type_specific=True):
        self.records, self.tokenizer, self.max_length, self.target_mode, self.type_specific = records, tokenizer, int(max_length), target_mode, type_specific
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        prompt, target = build_training_pair(self.records[idx], self.target_mode, self.type_specific)
        p = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        t = self.tokenizer(target, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]
        p, t = fit_prompt_target(p, t, self.max_length)
        return {"input_ids": p + t, "attention_mask": [1] * (len(p) + len(t)), "labels": [-100] * len(p) + t}

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID
    def __call__(self, batch):
        max_len = int(math.ceil(max(len(x["input_ids"]) for x in batch) / 8) * 8)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

def sample_records_by_type(records, per_type, seed=42):
    rng = random.Random(seed); by_type = defaultdict(list)
    for rec in records: by_type[rec["type"]].append(rec)
    out = []
    for items in by_type.values():
        items = list(items); rng.shuffle(items); out.extend(items[:per_type])
    rng.shuffle(out); return out

ablation_train_records = sample_records_by_type(train_records, TRAIN_SAMPLE_PER_TYPE_FOR_ABLATION, SEED)
ablation_valid_records = sample_records_by_type(valid_records, VALID_SAMPLE_PER_TYPE_FOR_ABLATION, SEED)
print("Ablation train:", len(ablation_train_records), "| ablation valid:", len(ablation_valid_records), "| final train:", len(train_records), "| final valid:", len(valid_records))

Tokenizer vocab_size: 50257 | len: 50258 | pad/eos: 50256 50256
Ablation train: 5600 | ablation valid: 960 | final train: 85543 | final valid: 3344


In [6]:
# 6. Rule solver deterministic
def normalize_query_for_rules(text):
    return str(text or "").lower().replace("−", "-").replace("–", "-").replace("—", "-").replace("×", "*").replace("÷", "/")

def query_numbers(text):
    text = normalize_query_for_rules(text)
    nums, spans = [], []
    for rx in [LATEX_FRAC_RE, SIMPLE_FRAC_RE]:
        for m in rx.finditer(text):
            val = parse_answer_number(m.group(0))
            if val is not None:
                nums.append((m.start(), val, m.group(0))); spans.append((m.start(), m.end()))
    for m in NUMBER_RE.finditer(text):
        if any(a <= m.start() < b for a, b in spans): continue
        val = parse_decimal(m.group(0))
        if val is not None: nums.append((m.start(), val, m.group(0)))
    return sorted(nums, key=lambda x: x[0])

def answer_with_conf(v, conf, reason):
    return canonical_answer_text(float(v)), float(conf), reason

def try_explicit_arithmetic(q):
    q = normalize_query_for_rules(q)
    if not re.search(r"(bằng bao nhiêu|tính|giá trị|kết quả)", q): return None
    exprs = re.findall(r"[-+]?\d+(?:[\.,]\d+)?(?:\s*[+\-*/^]\s*[-+]?\d+(?:[\.,]\d+)?)+", q)
    if not exprs: return None
    expr = re.sub(r"\d+(?:[\.,]\d+)?", lambda m: normalize_numeric_token(m.group(0)), exprs[-1])
    val = safe_eval_numeric_expr(expr)
    return answer_with_conf(val, 0.99, "explicit_arithmetic") if val is not None else None

def try_lcm_gcd(q):
    q = normalize_query_for_rules(q)
    nums = [int(round(v)) for _, v, _ in query_numbers(q) if abs(v - round(v)) < 1e-9 and v > 0]
    if len(nums) < 2: return None
    if re.search(r"\b(lcm|bội chung nhỏ nhất|bcnn)\b", q):
        if len(nums) % 2 == 0 and re.search(r"(cao nhất|lớn nhất|max|maximum)", q):
            vals = [abs(a*b)//math.gcd(a,b) for a,b in zip(nums[0::2], nums[1::2])]
            return answer_with_conf(max(vals), 0.98, "max_lcm_pairs")
        val = nums[0]
        for n in nums[1:]: val = abs(val*n)//math.gcd(val,n)
        return answer_with_conf(val, 0.96, "lcm")
    if re.search(r"\b(gcd|ước chung lớn nhất|ucln)\b", q):
        val = nums[0]
        for n in nums[1:]: val = math.gcd(val,n)
        return answer_with_conf(val, 0.96, "gcd")
    return None

def try_sequence(q):
    q = normalize_query_for_rules(q)
    if "..." not in q and "\\ldots" not in q and "danh sách" not in q and "dãy" not in q: return None
    nums = [v for _, v, _ in query_numbers(q)]
    if len(nums) < 4: return None
    d = nums[1] - nums[0]
    if abs(d) < 1e-12: return None
    n = (nums[-1] - nums[0]) / d + 1
    if abs(n - round(n)) <= 1e-6 and n > 0 and re.search(r"(bao nhiêu số|số lượng|có bao nhiêu)", q):
        return answer_with_conf(round(n), 0.97, "arithmetic_sequence_count")
    return None

def try_geometry(q):
    q = normalize_query_for_rules(q); nums = [v for _, v, _ in query_numbers(q)]
    if not nums: return None
    if "chu vi" in q:
        if "tam giác" in q and len(nums) >= 3: return answer_with_conf(sum(nums[:3]), 0.94, "triangle_perimeter")
        if "tam giác" in q and len(nums) == 2 and "hai cạnh bằng" in q: return answer_with_conf(nums[0]*2+nums[1], 0.97, "isosceles_triangle_perimeter")
        if "hình chữ nhật" in q and len(nums) >= 2: return answer_with_conf(2*(nums[0]+nums[1]), 0.95, "rectangle_perimeter")
        if "hình vuông" in q: return answer_with_conf(4*nums[0], 0.95, "square_perimeter")
    if "diện tích" in q:
        if "hình chữ nhật" in q and len(nums) >= 2: return answer_with_conf(nums[0]*nums[1], 0.95, "rectangle_area")
        if "hình vuông" in q: return answer_with_conf(nums[0]**2, 0.95, "square_area")
        if "tam giác" in q and len(nums) >= 2 and re.search(r"(đáy|chiều cao)", q): return answer_with_conf(nums[0]*nums[1]/2, 0.95, "triangle_area")
    if "thể tích" in q:
        if "lập phương" in q: return answer_with_conf(nums[0]**3, 0.95, "cube_volume")
        if len(nums) >= 3 and ("hộp" in q or "hình hộp" in q): return answer_with_conf(nums[0]*nums[1]*nums[2], 0.95, "box_volume")
    return None

def try_average_unit(q):
    q = normalize_query_for_rules(q); nums = [v for _, v, _ in query_numbers(q)]
    if not nums: return None
    if ("trung bình" in q or "average" in q) and re.search(r"10\s*k|10\s*km", q) and "phút" in q and len(nums) >= 3:
        mins = [x for x in nums if x != 10]
        if len(mins) >= 2: return answer_with_conf(sum(mins[:2])/10, 0.95, "race_average_time_per_km")
    if re.search(r"tá|dozen", q) and "bao nhiêu" in q: return answer_with_conf(nums[0]*12, 0.94, "dozen")
    if "giờ" in q and "phút" in q and re.search(r"đổi|chuyển|bao nhiêu phút", q): return answer_with_conf(nums[0]*60, 0.94, "hour_to_minute")
    return None

def solve_by_rules(query_vi, problem_type=None):
    q = str(query_vi or "")
    is_reverse = str(problem_type or "").endswith(("SV","FOBAR")) or re.search(r"giá trị của biến|giá trị biến|biến x", q, re.I)
    for fn in [try_explicit_arithmetic, try_lcm_gcd, try_sequence, try_geometry, try_average_unit]:
        try:
            res = fn(q)
        except Exception:
            res = None
        if res is not None:
            return res
    return None, 0.0, "no_rule"

for q,t in [("giải bài toán 1+1 bằng bao nhiêu?","GSM"), ("Một tam giác có hai cạnh bằng 7 và cạnh còn lại bằng 5. Chu vi tam giác là bao nhiêu?","MATH")]:
    print(q, "=>", solve_by_rules(q,t))

giải bài toán 1+1 bằng bao nhiêu? => ('2', 0.99, 'explicit_arithmetic')
Một tam giác có hai cạnh bằng 7 và cạnh còn lại bằng 5. Chu vi tam giác là bao nhiêu? => ('19', 0.97, 'isosceles_triangle_perimeter')


In [7]:
# 7. Retrieval TF-IDF char n-gram và tune threshold theo type
class RetrievalIndex:
    def __init__(self, records):
        self.records = records
        self.enabled = SKLEARN_OK and bool(records)
        if self.enabled:
            self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=300_000)
            self.matrix = self.vectorizer.fit_transform([r["query_vi"] for r in records])
            self.nn = NearestNeighbors(n_neighbors=min(5, len(records)), metric="cosine", algorithm="brute").fit(self.matrix)
            print("Retrieval index:", self.matrix.shape)
    def query(self, q, k=5):
        if not self.enabled: return []
        dist, idx = self.nn.kneighbors(self.vectorizer.transform([q]), n_neighbors=min(k, len(self.records)))
        return [{"similarity": 1.0-float(d), "record": self.records[int(i)], "answer": self.records[int(i)]["gold_answer"]} for d,i in zip(dist[0], idx[0])]
    def query_many(self, queries, k=1, batch_size=512):
        if not self.enabled: return [[] for _ in queries]
        out = []
        n_neighbors = min(k, len(self.records))
        for start in tqdm(range(0, len(queries), batch_size), desc="retrieval batch"):
            vec = self.vectorizer.transform(queries[start:start+batch_size])
            dist, idx = self.nn.kneighbors(vec, n_neighbors=n_neighbors)
            for drow, irow in zip(dist, idx):
                out.append([{"similarity": 1.0-float(d), "record": self.records[int(i)], "answer": self.records[int(i)]["gold_answer"]} for d,i in zip(drow, irow)])
        return out

def score_answer_pair(pred, gold):
    return score_from_rel(relative_error(parse_answer_number(pred), parse_answer_number(gold)))

def tune_retrieval_thresholds(index, valid_records, default=0.92):
    if not getattr(index, "enabled", False): return {}, []
    rows = []
    hits_many = index.query_many([rec["query_vi"] for rec in valid_records], k=1)
    for rec, hits in zip(valid_records, hits_many):
        if not hits:
            continue
        hit = hits[0]
        rows.append(dict(id=rec["id"], type=rec["type"], similarity=hit["similarity"], pred_answer=hit["answer"], gold_answer=rec["gold_answer"], score=score_answer_pair(hit["answer"], rec["gold_answer"])))
    thresholds = {}
    for typ in sorted(set(r["type"] for r in rows)):
        sub = [r for r in rows if r["type"] == typ]
        best = (default, -1, 0.0, 0.0)
        for thr in np.arange(default, 0.991, 0.01):
            chosen = [r for r in sub if r["similarity"] >= thr]
            raw = sum(r["score"] for r in chosen)
            precision = float(np.mean([r["score"] == 10 for r in chosen])) if chosen else 0.0
            coverage = len(chosen) / max(1, len(sub))
            if (raw, precision, coverage) > (best[1], best[2], best[3]):
                best = (round(float(thr), 2), raw, precision, coverage)
        thresholds[typ] = best[0]
    return thresholds, rows

retrieval_index = RetrievalIndex(train_records)
retrieval_thresholds, retrieval_tuning_rows = tune_retrieval_thresholds(retrieval_index, valid_records, RETRIEVAL_DEFAULT_THRESHOLD)
print("Retrieval thresholds:", retrieval_thresholds)

Retrieval index: (85543, 146650)


retrieval batch:   0%|          | 0/7 [00:00<?, ?it/s]

Retrieval thresholds: {'GSM_AnsAug': 0.92, 'GSM_FOBAR': 0.92, 'GSM_Rephrased': 0.92, 'GSM_SV': 0.92, 'MATH_AnsAug': 0.92, 'MATH_FOBAR': 0.92, 'MATH_Rephrased': 0.92, 'MATH_SV': 0.92}


In [8]:
# 8. LoRA/fallback train functions
def ensure_model_config(model):
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    if hasattr(model, "generation_config"):
        model.generation_config.pad_token_id = SAFE_EOS_ID
        model.generation_config.eos_token_id = SAFE_EOS_ID
    if len(tokenizer) > model.get_input_embeddings().num_embeddings:
        model.resize_token_embeddings(len(tokenizer))
    return model

def count_trainable_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return dict(trainable=int(tr), total=int(total), pct=float(tr/max(1,total)))

def load_trainable_model():
    dtype = torch.float16 if CUDA_OK else torch.float32
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), torch_dtype=dtype, local_files_only=True)
    model = ensure_model_config(model)
    if hasattr(model, "gradient_checkpointing_enable"): model.gradient_checkpointing_enable()
    model.config.use_cache = False
    if PEFT_OK:
        cfg = LoraConfig(**LORA_CONFIG)
        model = get_peft_model(model, cfg)
        model.print_trainable_parameters()
        return model, "peft_lora"
    for p in model.parameters(): p.requires_grad = False
    n_layer = int(getattr(model.config, "n_layer", 12))
    train_ids = {n_layer-1, n_layer-2}
    for name, p in model.named_parameters():
        if name.startswith("lm_head") or any(f"transformer.h.{i}." in name for i in train_ids):
            p.requires_grad = True
    print("Fallback trainable:", count_trainable_parameters(model))
    return model, "partial_freeze_upper_layers"

def make_training_args(run_dir, cfg, quick=False):
    use_bf16 = bool(CUDA_OK and torch.cuda.is_bf16_supported())
    use_fp16 = bool(CUDA_OK and not use_bf16)
    save_eval = "no" if quick else "epoch"
    kwargs = dict(output_dir=str(run_dir/"trainer_tmp"), num_train_epochs=cfg["num_train_epochs"], per_device_train_batch_size=cfg["per_device_train_batch_size"], per_device_eval_batch_size=cfg["per_device_eval_batch_size"], gradient_accumulation_steps=cfg["gradient_accumulation_steps"], learning_rate=cfg["learning_rate"], warmup_ratio=cfg["warmup_ratio"], lr_scheduler_type="cosine", weight_decay=cfg["weight_decay"], logging_steps=cfg["logging_steps"], save_strategy=save_eval, save_total_limit=2, load_best_model_at_end=not quick, metric_for_best_model="eval_loss", greater_is_better=False, report_to="none", seed=SEED, data_seed=SEED, remove_unused_columns=False, dataloader_num_workers=2 if IS_KAGGLE else 0, gradient_checkpointing=True, max_grad_norm=1.0)
    if cfg.get("max_steps") is not None: kwargs["max_steps"] = int(cfg["max_steps"])
    sig = inspect.signature(TrainingArguments.__init__)
    kwargs["eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"] = save_eval
    if "bf16" in sig.parameters: kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters: kwargs["fp16"] = use_fp16
    if not CUDA_OK and "use_cpu" in sig.parameters: kwargs["use_cpu"] = True
    return TrainingArguments(**kwargs)

def train_experiment(exp_name, train_subset, valid_subset, target_mode, type_specific, max_length, train_cfg, quick=False):
    run_dir = RUNS_DIR / exp_name; run_dir.mkdir(parents=True, exist_ok=True)
    artifact_path = run_dir / ("adapter" if PEFT_OK else "full_model")
    metadata_path = run_dir / "metadata.json"
    if artifact_path.exists() and metadata_path.exists():
        print("Skip existing:", exp_name)
        return json.loads(metadata_path.read_text(encoding="utf-8"))
    if not CUDA_OK and not ALLOW_CPU_TRAINING:
        metadata = dict(exp_name=exp_name, run_dir=str(run_dir), artifact_path=None, model_kind=None, skipped="no_cuda")
        save_json(metadata, metadata_path); print("Skip train no CUDA:", exp_name); return metadata
    train_ds = MathSFTDataset(train_subset, tokenizer, max_length, target_mode, type_specific)
    eval_subset = valid_subset[:TRAINER_EVAL_SAMPLES]
    eval_ds = MathSFTDataset(eval_subset, tokenizer, max_length, target_mode, type_specific)
    model, kind = load_trainable_model()
    trainer = Trainer(model=model, args=make_training_args(run_dir, train_cfg, quick=quick), train_dataset=train_ds, eval_dataset=None if quick else eval_ds, data_collator=PadCollator())
    t0 = time.time(); result = trainer.train(); minutes = (time.time() - t0) / 60
    trainer.model.save_pretrained(str(artifact_path)); tokenizer.save_pretrained(str(artifact_path))
    metadata = dict(exp_name=exp_name, run_dir=str(run_dir), artifact_path=str(artifact_path), model_kind=kind, target_mode=target_mode, type_specific=type_specific, max_length=max_length, train_samples=len(train_subset), eval_samples_for_trainer=len(eval_subset), train_minutes=minutes, train_output=str(result), trainable_parameters=count_trainable_parameters(trainer.model), peft_available=PEFT_OK, lora_config=LORA_CONFIG if PEFT_OK else None, training_args=train_cfg)
    save_json(metadata, metadata_path)
    try: trainer.state.save_to_json(str(run_dir/"trainer_state.json"))
    except Exception: pass
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return metadata

In [9]:
# 9. Generation, voting, predict pipeline
def load_generation_model(artifact):
    if not artifact or artifact.get("artifact_path") is None: return None, None
    device = "cuda" if CUDA_OK else "cpu"; dtype = torch.float16 if device == "cuda" else torch.float32
    gen_tok = AutoTokenizer.from_pretrained(artifact["artifact_path"], local_files_only=True)
    gen_tok.pad_token_id = SAFE_EOS_ID; gen_tok.eos_token_id = SAFE_EOS_ID
    if getattr(gen_tok, "pad_token", None) is None and getattr(gen_tok, "eos_token", None) is not None: gen_tok.pad_token = gen_tok.eos_token
    gen_tok.padding_side = "left"
    gen_tok.truncation_side = "left"
    if artifact.get("model_kind") == "peft_lora" and PEFT_OK:
        base = ensure_model_config(AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), torch_dtype=dtype, local_files_only=True))
        model = PeftModel.from_pretrained(base, artifact["artifact_path"], local_files_only=True)
    else:
        model = ensure_model_config(AutoModelForCausalLM.from_pretrained(artifact["artifact_path"], torch_dtype=dtype, local_files_only=True))
    model.to(device); model.eval(); model.config.use_cache = True
    return model, gen_tok

def stop_tail(text):
    text = str(text or "").strip()
    for marker in ["\nNhiệm vụ:", "\nBài toán:", "\nCâu hỏi:", "\nQuestion:", "###"]:
        pos = text.find(marker)
        if pos >= 0: text = text[:pos].strip()
    return text

def extract_from_candidate(prompt, tail):
    tail = stop_tail(tail)
    return extract_final_answer("Đáp án:" + tail) or extract_final_answer(tail) or extract_final_answer(prompt + tail)

def vote_answers(candidates, tie_breaker=None):
    votes, display_ans = Counter(), {}
    for c in candidates:
        val = parse_answer_number(c.get("answer"))
        if val is None:
            continue
        key = canonical_answer_text(val)
        weight = int(c.get("weight", 1))
        votes[key] += max(1, weight)
        display_ans.setdefault(key, canonical_answer_text(val, c.get("answer")))
    if not votes:
        return None, dict(votes={}, reason="no_extractable_answer")
    tb_val = parse_answer_number(tie_breaker)
    tb_key = canonical_answer_text(tb_val) if tb_val is not None else None
    max_count = max(votes.values())
    tied = [k for k, v in votes.items() if v == max_count]
    chosen = tb_key if tb_key in tied else tied[0]
    return display_ans[chosen], dict(votes=dict(votes), chosen_key=chosen, reason="majority" if len(tied) == 1 else "majority_tie")

def generate_candidate_batches(model, gen_tok, prompts, num_return_sequences=1, do_sample=False, temperature=0.7, top_p=0.95, max_new_tokens=48, batch_size=8, num_beams=1, candidate_source="model"):
    out = [[] for _ in prompts]
    if not prompts: return out
    device = next(model.parameters()).device
    with torch.inference_mode():
        for start in tqdm(range(0, len(prompts), batch_size), desc="generate"):
            batch = prompts[start:start+batch_size]
            enc = gen_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH_SHORT).to(device)
            kwargs = dict(input_ids=enc["input_ids"], attention_mask=enc.get("attention_mask"), max_new_tokens=max_new_tokens, do_sample=do_sample, num_return_sequences=num_return_sequences, num_beams=num_beams, repetition_penalty=1.05, pad_token_id=SAFE_EOS_ID, eos_token_id=SAFE_EOS_ID)
            if do_sample:
                kwargs.update(temperature=temperature, top_p=top_p)
            gen = model.generate(**kwargs)
            prompt_len = enc["input_ids"].shape[1]
            decoded = gen_tok.batch_decode(gen[:, prompt_len:], skip_special_tokens=True)
            for i in range(len(batch)):
                for tail in decoded[i*num_return_sequences:(i+1)*num_return_sequences]:
                    out[start+i].append(dict(tail=stop_tail(tail), answer=extract_from_candidate(batch[i], tail), source=candidate_source))
    return out

def format_model_output(answer):
    answer = str(answer if answer is not None else "0").strip()
    return f"Lời giải: Tính theo dữ kiện trong đề.\nĐáp án là: {answer}"

def make_output_record(rec, answer, source="model", detail=None):
    return dict(id=rec.get("id"), query_vi=rec.get("query_vi",""), type=rec.get("type","unknown"), model_output=format_model_output(answer), _source=source, _detail=detail or {})

def public_output(outputs):
    return [{k:o[k] for k in ["id","query_vi","type","model_output"]} for o in outputs]

def evaluate_outputs_against_records(outputs, records):
    by_id = {str(o["id"]): o for o in outputs}
    ordered = [by_id[str(r["id"])] for r in records]
    rows = evaluate_answer_texts([o["model_output"] for o in ordered], [r["response_vi"] for r in records], [r["type"] for r in records], [r["id"] for r in records])
    return rows, summarize_eval_rows(rows)

def summarize_retrieval_hits(hits, threshold):
    if not hits:
        return None
    top = hits[0]
    top_sim = top["similarity"]
    close_hits = [h for h in hits if top_sim - h["similarity"] <= RETRIEVAL_CLOSE_DELTA]
    answer_votes, display_answer = Counter(), {}
    for hit in close_hits:
        val = parse_answer_number(hit.get("answer"))
        if val is None:
            continue
        key = canonical_answer_text(val)
        answer_votes[key] += 1
        display_answer.setdefault(key, canonical_answer_text(val, hit.get("answer")))
    if answer_votes:
        max_vote = max(answer_votes.values())
        tied = [k for k, v in answer_votes.items() if v == max_vote]
        voted_key = tied[0]
        voted_answer = display_answer[voted_key]
        vote_tie = len(tied) > 1
    else:
        voted_answer = top["answer"]
        vote_tie = False
    return dict(
        hit=top,
        hits=hits,
        answer=voted_answer,
        similarity=top_sim,
        threshold=threshold,
        close_count=len(close_hits),
        vote_counts=dict(answer_votes),
        vote_tie=vote_tie,
        use_direct=(top_sim >= threshold and not vote_tie),
        use_fewshot=top_sim >= RETRIEVAL_FEWSHOT_THRESHOLD,
    )

def get_retrieval_hint(rec, threshold_map=None, k=None):
    k = RETRIEVAL_TOP_K if k is None else k
    hits = retrieval_index.query(rec["query_vi"], k) if getattr(retrieval_index, "enabled", False) else []
    threshold = (threshold_map or {}).get(rec.get("type"), RETRIEVAL_DEFAULT_THRESHOLD)
    return summarize_retrieval_hits(hits, threshold)

def get_retrieval_hints(records, threshold_map=None, k=None):
    if not getattr(retrieval_index, "enabled", False):
        return [None] * len(records)
    k = RETRIEVAL_TOP_K if k is None else k
    hits_many = retrieval_index.query_many([r["query_vi"] for r in records], k=k)
    hints = []
    for rec, hits in zip(records, hits_many):
        threshold = (threshold_map or {}).get(rec.get("type"), RETRIEVAL_DEFAULT_THRESHOLD)
        hints.append(summarize_retrieval_hits(hits, threshold))
    return hints

def should_accept_rule(confidence, reason):
    return confidence >= RULE_HIGH_CONFIDENCE and (not RULE_REASON_ALLOWLIST or reason in RULE_REASON_ALLOWLIST)

def build_generation_prompt_for_artifact(rec, artifact=None, type_specific=True, fewshot=None):
    target_mode = (artifact or {}).get("target_mode")
    if target_mode in {"original", "short_solution", "calc_tail"}:
        prompt = build_solution_prompt(rec, type_specific)
        if fewshot:
            prompt = (
                "Ví dụ gần giống:\n"
                f"Bài toán: {fewshot['query_vi'].strip()}\n"
                f"Lời giải:\nĐáp án là: {fewshot['gold_answer']}\n\n"
                + prompt
            )
        return prompt
    return build_answer_prompt(rec, type_specific, fewshot)

def predict_one(example, model_context=None, threshold_map=None, artifact=None):
    ans, conf, reason = solve_by_rules(example["query_vi"], example.get("type"))
    if should_accept_rule(conf, reason):
        return make_output_record(example, ans, "rule", dict(confidence=conf, reason=reason))
    hint = get_retrieval_hint(example, threshold_map)
    if hint and hint["use_direct"]:
        return make_output_record(example, hint["answer"], "retrieval", dict(similarity=hint["similarity"], threshold=hint["threshold"], vote_counts=hint.get("vote_counts"), close_count=hint.get("close_count")))
    if model_context is None:
        fallback = ans if ans is not None else (hint["answer"] if hint else "0")
        return make_output_record(example, fallback, "fallback_no_model")
    model, gen_tok = model_context
    fewshot = hint["hit"]["record"] if hint and hint["use_fewshot"] else None
    prompt = build_generation_prompt_for_artifact(example, artifact, True, fewshot)
    greedy = generate_candidate_batches(model, gen_tok, [prompt], 1, False, max_new_tokens=GENERATION["fallback_max_new_tokens"], batch_size=1, num_beams=3, candidate_source="greedy_beam")[0]
    for cand in greedy:
        cand["weight"] = 2
    sampled = generate_candidate_batches(model, gen_tok, [prompt], GENERATION["num_return_sequences"], True, GENERATION["temperature"], GENERATION["top_p"], GENERATION["max_new_tokens"], 1, candidate_source="sample")[0]
    cands = greedy + sampled
    voted, detail = vote_answers(cands, tie_breaker=ans if conf >= 0.7 else None)
    return make_output_record(example, voted if voted is not None else "0", "model_voting", detail)

def predict_records_pipeline(records, artifact=None, use_rules=True, use_retrieval=True, use_voting=True, type_specific=True, threshold_map=None, name="predict"):
    outputs, fallback_records, fallback_prompts, fallback_hints = [], [], [], []
    threshold_map = threshold_map or retrieval_thresholds
    batch_hints = get_retrieval_hints(records, threshold_map) if use_retrieval else [None] * len(records)
    for rec, hint in zip(records, batch_hints):
        r_ans, r_conf, r_reason = solve_by_rules(rec["query_vi"], rec.get("type")) if use_rules else (None,0.0,"rules_off")
        if use_rules and should_accept_rule(r_conf, r_reason):
            outputs.append(make_output_record(rec, r_ans, "rule", dict(confidence=r_conf, reason=r_reason))); continue
        if use_retrieval and hint and hint["use_direct"]:
            outputs.append(make_output_record(rec, hint["answer"], "retrieval", dict(similarity=hint["similarity"], threshold=hint["threshold"], vote_counts=hint.get("vote_counts"), close_count=hint.get("close_count")))); continue
        fewshot = hint["hit"]["record"] if hint and hint["use_fewshot"] else None
        fallback_records.append(rec); fallback_prompts.append(build_generation_prompt_for_artifact(rec, artifact, type_specific, fewshot)); fallback_hints.append(dict(rule_ans=r_ans, rule_conf=r_conf, hint=hint))
    print(f"{name}: direct={len(outputs)} model_fallback={len(fallback_records)}")
    if fallback_records:
        if not artifact or artifact.get("artifact_path") is None:
            for rec, h in zip(fallback_records, fallback_hints):
                ans = h["rule_ans"] if h["rule_ans"] is not None else (h["hint"]["answer"] if h.get("hint") else "0")
                outputs.append(make_output_record(rec, ans, "fallback_no_model"))
        else:
            model, gen_tok = load_generation_model(artifact)
            try:
                if use_voting:
                    greedy_by = generate_candidate_batches(model, gen_tok, fallback_prompts, 1, False, max_new_tokens=GENERATION["fallback_max_new_tokens"], batch_size=GENERATION["batch_size"], num_beams=3, candidate_source="greedy_beam")
                    sample_by = generate_candidate_batches(model, gen_tok, fallback_prompts, GENERATION["num_return_sequences"], True, GENERATION["temperature"], GENERATION["top_p"], GENERATION["max_new_tokens"], GENERATION["batch_size"], candidate_source="sample")
                    cands_by = []
                    for greedy, sampled in zip(greedy_by, sample_by):
                        for cand in greedy:
                            cand["weight"] = 2
                        cands_by.append(greedy + sampled)
                else:
                    cands_by = generate_candidate_batches(model, gen_tok, fallback_prompts, 1, False, max_new_tokens=GENERATION["fallback_max_new_tokens"], batch_size=GENERATION["batch_size"], num_beams=3, candidate_source="greedy_beam")
                retry_prompts, retry_idx, answers, details = [], [], [], []
                for i, (cands, h) in enumerate(zip(cands_by, fallback_hints)):
                    tie = h["rule_ans"] if h["rule_conf"] >= 0.7 else (h["hint"]["answer"] if h.get("hint") and h["hint"]["similarity"] >= 0.85 else None)
                    ans, detail = vote_answers(cands, tie)
                    answers.append(ans); details.append(detail)
                    if ans is None: retry_prompts.append(fallback_prompts[i]); retry_idx.append(i)
                if retry_prompts:
                    retry = generate_candidate_batches(model, gen_tok, retry_prompts, 1, False, max_new_tokens=GENERATION["fallback_max_new_tokens"], batch_size=GENERATION["batch_size"], num_beams=3, candidate_source="retry_greedy_beam")
                    for i, cands in zip(retry_idx, retry):
                        ans, detail = vote_answers(cands)
                        if ans is not None: answers[i], details[i] = ans, dict(retry=True, **detail)
                for rec, ans, detail in zip(fallback_records, answers, details):
                    outputs.append(make_output_record(rec, ans if ans is not None else "0", "model_voting" if use_voting else "model_greedy", detail))
            finally:
                del model, gen_tok; gc.collect(); torch.cuda.empty_cache()
    by_id = {str(o["id"]): o for o in outputs}
    return [by_id[str(r["id"])] for r in records]

In [10]:
# 10. Ablation experiments
experiment_reports, artifacts = [], {}

def grouped_output_summary(rows, outputs, group_getter):
    groups = defaultdict(list)
    by_id = {str(o["id"]): o for o in outputs}
    for row in rows:
        out = by_id[str(row["id"])]
        groups[group_getter(out)].append(row)
    return {str(k): summarize_eval_rows_no_type(v) for k, v in sorted(groups.items(), key=lambda kv: str(kv[0]))}

def add_experiment_report(name, outputs, records, notes=None):
    rows, summary = evaluate_outputs_against_records(outputs, records)
    by_id = {str(o["id"]): o for o in outputs}
    source_summary = grouped_output_summary(rows, outputs, lambda out: out.get("_source", "unknown"))
    rule_reason_summary = grouped_output_summary(rows, outputs, lambda out: out.get("_detail", {}).get("reason", "not_rule") if out.get("_source") == "rule" else "not_rule")
    errors = []
    for row, rec in zip(rows, records):
        if row["score"] < 10:
            out = by_id[str(rec["id"])]
            errors.append(dict(id=rec["id"], type=rec["type"], score=row["score"], relative_error=row["relative_error"], pred_answer=row["pred_answer"], gold_answer=row["gold_answer"], source=out.get("_source"), detail=out.get("_detail"), query_vi=rec["query_vi"][:500], model_output=out["model_output"][:500]))
    report = dict(name=name, summary=summary, source_summary=source_summary, rule_reason_summary=rule_reason_summary, notes=notes or {}, top_error_cases=errors[:30])
    experiment_reports.append(report)
    print("\n", name, "score_10=", round(summary["score_10"], 4), "not_extractable=", summary["not_extractable_count"])
    display(pd.DataFrame(summary["by_type"]).T[["num_samples","score_10","pass_1pct_rate","pass_10pct_rate","not_extractable_count"]])
    return report

baseline_outputs = [make_output_record(r, "0", "baseline_constant_zero") for r in ablation_valid_records]
add_experiment_report("Baseline hiện tại", baseline_outputs, ablation_valid_records, {"baseline": "constant_zero_format_check; bật thêm base generation nếu cần"})

if RUN_QUICK_ABLATIONS:
    artifacts["lora_original_generic"] = train_experiment("lora_original_generic", ablation_train_records, ablation_valid_records, "original", False, MAX_LENGTH_ORIGINAL, ABLATION_TRAINING, quick=True)
    out = predict_records_pipeline(ablation_valid_records, artifacts["lora_original_generic"], False, False, False, False, name="lora_original_generic")
    add_experiment_report("LoRA + target gốc", out, ablation_valid_records, {"target": "response_vi original", "prompt": "generic"})

    artifacts["lora_answer_only_generic"] = train_experiment("lora_answer_only_generic", ablation_train_records, ablation_valid_records, "answer_only", False, MAX_LENGTH_SHORT, ABLATION_TRAINING, quick=True)
    out = predict_records_pipeline(ablation_valid_records, artifacts["lora_answer_only_generic"], False, False, False, False, name="lora_answer_only_generic")
    add_experiment_report("LoRA + answer-only target", out, ablation_valid_records, {"target": "answer_only", "prompt": "generic"})

    artifacts["lora_answer_only_type_quick"] = train_experiment("lora_answer_only_type_quick", ablation_train_records, ablation_valid_records, "answer_only", True, MAX_LENGTH_SHORT, ABLATION_TRAINING, quick=True)
    out = predict_records_pipeline(ablation_valid_records, artifacts["lora_answer_only_type_quick"], False, False, False, True, name="lora_answer_only_type")
    add_experiment_report("LoRA + answer-only + type-specific prompt", out, ablation_valid_records, {"target": "answer_only", "prompt": "TYPE_INSTRUCTION_MAP"})

    artifacts["lora_calc_tail_type_quick"] = train_experiment("lora_calc_tail_type_quick", ablation_train_records, ablation_valid_records, "calc_tail", True, MAX_LENGTH_SHORT, ABLATION_TRAINING, quick=True)
    out = predict_records_pipeline(ablation_valid_records, artifacts["lora_calc_tail_type_quick"], False, False, False, True, name="lora_calc_tail_type")
    add_experiment_report("LoRA + calc-tail + type-specific prompt", out, ablation_valid_records, {"target": "calc_tail", "prompt": "TYPE_INSTRUCTION_MAP"})

    out = predict_records_pipeline(ablation_valid_records, artifacts["lora_answer_only_type_quick"], False, False, True, True, name="lora_answer_only_type_voting")
    add_experiment_report("LoRA + answer-only + prompt + voting", out, ablation_valid_records, {"target": "answer_only", "generation": GENERATION})

    out = predict_records_pipeline(ablation_valid_records, artifacts["lora_calc_tail_type_quick"], False, False, True, True, name="lora_calc_tail_type_voting")
    add_experiment_report("LoRA + calc-tail + prompt + voting", out, ablation_valid_records, {"target": "calc_tail", "generation": GENERATION})
else:
    print("RUN_QUICK_ABLATIONS=False: skip GPU ablation training in local dry-run.")

def choose_final_target_mode():
    if FINAL_TARGET_MODE != "auto":
        return FINAL_TARGET_MODE, {"selection": "manual", "target_mode": FINAL_TARGET_MODE}
    candidates = []
    for report in experiment_reports:
        target = report.get("notes", {}).get("target")
        if target in {"answer_only", "calc_tail"} and "voting" in report.get("name", "").lower():
            candidates.append((report["summary"]["score_10"], target, report["name"]))
    if candidates:
        score, target, name = max(candidates, key=lambda x: x[0])
        return target, {"selection": "best_ablation_valid_score", "selected_report": name, "score_10": score}
    return "answer_only", {"selection": "fallback_no_ablation", "target_mode": "answer_only"}

FINAL_SELECTED_TARGET_MODE, FINAL_SELECTION_INFO = choose_final_target_mode()
print("FINAL_SELECTED_TARGET_MODE:", FINAL_SELECTED_TARGET_MODE, FINAL_SELECTION_INFO)



 Baseline hiện tại score_10= 0.3615 not_extractable= 0


,num_samples,score_10,pass_1pct_rate,pass_10pct_rate,not_extractable_count
GSM_AnsAug,120.0,0.000000,0.000000,0.000000,0.0
GSM_FOBAR,120.0,0.000000,0.000000,0.000000,0.0
GSM_Rephrased,120.0,0.000000,0.000000,0.000000,0.0
GSM_SV,120.0,0.000000,0.000000,0.000000,0.0
MATH_AnsAug,120.0,0.391667,0.025000,0.050000,0.0
MATH_FOBAR,120.0,0.500000,0.050000,0.050000,0.0
MATH_Rephrased,120.0,1.166667,0.083333,0.141667,0.0
MATH_SV,120.0,0.833333,0.083333,0.083333,0.0


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


...

generate:   0%|          | 0/120 [00:00<?, ?it/s]

generate:   0%|          | 0/120 [00:00<?, ?it/s]


 LoRA + calc-tail + prompt + voting score_10= 0.9062 not_extractable= 0


,num_samples,score_10,pass_1pct_rate,pass_10pct_rate,not_extractable_count
GSM_AnsAug,120.0,0.375000,0.016667,0.025000,0.0
GSM_FOBAR,120.0,0.458333,0.016667,0.033333,0.0
GSM_Rephrased,120.0,0.666667,0.033333,0.058333,0.0
GSM_SV,120.0,0.558333,0.033333,0.050000,0.0
MATH_AnsAug,120.0,0.433333,0.008333,0.050000,0.0
MATH_FOBAR,120.0,2.083333,0.175000,0.183333,0.0
MATH_Rephrased,120.0,0.708333,0.025000,0.075000,0.0
MATH_SV,120.0,1.966667,0.175000,0.175000,0.0


FINAL_SELECTED_TARGET_MODE: answer_only {'selection': 'best_ablation_valid_score', 'selected_report': 'LoRA + answer-only + prompt + voting', 'score_10': 1.2125}


In [11]:
# 11. Final train + full validation pipeline
FINAL_ARTIFACT_KEY = f"final_{FINAL_SELECTED_TARGET_MODE}_type"
FINAL_EXPERIMENT_NAME = FINAL_ARTIFACT_KEY
if RUN_FINAL_TRAIN:
    artifacts[FINAL_ARTIFACT_KEY] = train_experiment(FINAL_EXPERIMENT_NAME, train_records, valid_records, FINAL_SELECTED_TARGET_MODE, True, MAX_LENGTH_SHORT, FINAL_TRAINING, quick=False)
else:
    quick_key = "lora_calc_tail_type_quick" if FINAL_SELECTED_TARGET_MODE == "calc_tail" else "lora_answer_only_type_quick"
    artifacts[FINAL_ARTIFACT_KEY] = artifacts.get(quick_key, {"artifact_path": None, "model_kind": None, "target_mode": FINAL_SELECTED_TARGET_MODE, "skipped": "no_final_train"})
    print("RUN_FINAL_TRAIN=False, final artifact:", artifacts[FINAL_ARTIFACT_KEY])

final_outputs_debug = predict_records_pipeline(valid_records, artifacts[FINAL_ARTIFACT_KEY], True, True, True, True, retrieval_thresholds, name="full_pipeline_valid")
final_report = add_experiment_report("Full pipeline: rule + retrieval + LoRA + voting", final_outputs_debug, valid_records, dict(target=FINAL_SELECTED_TARGET_MODE, prompt="artifact-aware TYPE_INSTRUCTION_MAP", rule_high_confidence=RULE_HIGH_CONFIDENCE, rule_reason_allowlist=sorted(RULE_REASON_ALLOWLIST), retrieval_thresholds=retrieval_thresholds, retrieval_top_k=RETRIEVAL_TOP_K, generation=GENERATION, final_selection=FINAL_SELECTION_INFO, artifact=artifacts.get(FINAL_ARTIFACT_KEY)))

save_json(public_output(final_outputs_debug), VALID_OUTPUT_PATH)
print("Saved", VALID_OUTPUT_PATH, "records:", len(final_outputs_debug))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 2,359,296 || all params: 126,799,872 || trainable%: 1.8606


Epoch,Training Loss,Validation Loss
1,2.180592,2.162811
2,1.904944,2.242259
3,1.780020,2.298584
4,1.645669,2.370345


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

...

generate:   0%|          | 0/376 [00:00<?, ?it/s]

generate:   0%|          | 0/376 [00:00<?, ?it/s]


 Full pipeline: rule + retrieval + LoRA + voting score_10= 1.1839 not_extractable= 0


,num_samples,score_10,pass_1pct_rate,pass_10pct_rate,not_extractable_count
GSM_AnsAug,596.0,0.605705,0.026846,0.045302,0.0
GSM_FOBAR,423.0,1.408983,0.115839,0.115839,0.0
GSM_Rephrased,416.0,0.639423,0.028846,0.052885,0.0
GSM_SV,353.0,1.167139,0.093484,0.096317,0.0
MATH_AnsAug,559.0,1.093023,0.071556,0.096601,0.0
MATH_FOBAR,339.0,1.923304,0.156342,0.159292,0.0
MATH_Rephrased,384.0,1.151042,0.078125,0.098958,0.0
MATH_SV,274.0,2.259124,0.197080,0.197080,0.0


Saved /kaggle/working/valid_output.json records: 3344


In [12]:
# 12. Save valid_report.json
rows, summary = evaluate_outputs_against_records(final_outputs_debug, valid_records)
by_id = {str(o["id"]): o for o in final_outputs_debug}
top_errors = []
for row, rec in zip(rows, valid_records):
    if row["score"] < 10:
        out = by_id[str(rec["id"])]
        top_errors.append(dict(id=rec["id"], type=rec["type"], score=row["score"], relative_error=row["relative_error"], pred_answer=row["pred_answer"], gold_answer=row["gold_answer"], source=out.get("_source"), detail=out.get("_detail"), query_vi=rec["query_vi"][:800], gold_response_tail=rec["response_vi"][-500:], model_output=out["model_output"][:800]))

final_source_summary = grouped_output_summary(rows, final_outputs_debug, lambda out: out.get("_source", "unknown"))
final_rule_reason_summary = grouped_output_summary(rows, final_outputs_debug, lambda out: out.get("_detail", {}).get("reason", "not_rule") if out.get("_source") == "rule" else "not_rule")

valid_report_payload = dict(
    created_at=time.strftime("%Y-%m-%d %H:%M:%S"),
    data=dict(train_file=str(TRAIN_FILE), valid_file_buggy_loaded_but_unused=str(VALID_FILE_BUGGY) if VALID_FILE_BUGGY else None, split_report=split_report, train_records=len(train_records), valid_records=len(valid_records), gold_extract_failed=len(extract_fail), schema_note="normalize query/response hoặc query_vi/response_vi; gold target chỉ extract từ response_vi"),
    final_configuration=dict(backbone="NlpHUST/gpt2-vietnamese", model_dir=str(MODEL_DIR), selected_target_mode=FINAL_SELECTED_TARGET_MODE, final_selection=FINAL_SELECTION_INFO, target_format="selected target mode; public output always 'Lời giải... Đáp án là: <number>'.", lora_config=LORA_CONFIG if PEFT_OK else None, fallback_if_no_peft="freeze lower layers, train 2 upper GPT-2 blocks + lm_head", training_args=FINAL_TRAINING, prompt_template="artifact-aware: answer-only uses Đáp án:, calc-tail uses Lời giải:", type_instruction_map=TYPE_INSTRUCTION_MAP, decoding_voting=GENERATION, rule_solver="deterministic rules, high-confidence plus allowlist only", rule_reason_allowlist=sorted(RULE_REASON_ALLOWLIST), retrieval=dict(vectorizer="TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5))", thresholds_by_type=retrieval_thresholds, fewshot_threshold=RETRIEVAL_FEWSHOT_THRESHOLD, top_k=RETRIEVAL_TOP_K, close_delta=RETRIEVAL_CLOSE_DELTA)),
    experiments=experiment_reports,
    final_valid_summary=summary,
    final_source_counts=dict(Counter(o.get("_source") for o in final_outputs_debug)),
    final_source_summary=final_source_summary,
    final_rule_reason_summary=final_rule_reason_summary,
    retrieval_tuning_sample_count=len(retrieval_tuning_rows),
    top_error_cases=top_errors[:80],
)
save_json(valid_report_payload, VALID_REPORT_PATH)
print("Saved", VALID_REPORT_PATH)
print("Final score_10:", summary["score_10"], "| pct max:", summary["score_pct_of_max"])
display(pd.DataFrame(summary["by_type"]).T[["num_samples","score_10","pass_1pct_rate","pass_10pct_rate","pass_50pct_rate","not_extractable_count"]])
print("Score by source:")
display(pd.DataFrame(final_source_summary).T[["num_samples", "score_10", "pass_1pct_rate", "pass_10pct_rate", "not_extractable_count"]])
print("Rule reason summary:")
display(pd.DataFrame(final_rule_reason_summary).T[["num_samples", "score_10", "pass_1pct_rate", "pass_10pct_rate", "not_extractable_count"]])


Saved /kaggle/working/valid_report.json
Final score_10: 1.1839114832535884 | pct max: 0.11839114832535885


,num_samples,score_10,pass_1pct_rate,pass_10pct_rate,pass_50pct_rate,not_extractable_count
GSM_AnsAug,596.0,0.605705,0.026846,0.045302,0.290268,0.0
GSM_FOBAR,423.0,1.408983,0.115839,0.115839,0.366430,0.0
GSM_Rephrased,416.0,0.639423,0.028846,0.052885,0.283654,0.0
GSM_SV,353.0,1.167139,0.093484,0.096317,0.314448,0.0
MATH_AnsAug,559.0,1.093023,0.071556,0.096601,0.348837,0.0
MATH_FOBAR,339.0,1.923304,0.156342,0.159292,0.504425,0.0
MATH_Rephrased,384.0,1.151042,0.078125,0.098958,0.364583,0.0
MATH_SV,274.0,2.259124,0.197080,0.197080,0.485401,0.0


Score by source:


,num_samples,score_10,pass_1pct_rate,pass_10pct_rate,not_extractable_count
model_voting,3001.0,1.187271,0.085305,0.098967,0.0
retrieval,82.0,2.475610,0.207317,0.256098,0.0
rule,261.0,0.739464,0.053640,0.053640,0.0


Rule reason summary:


,num_samples,score_10,pass_1pct_rate,pass_10pct_rate,not_extractable_count
arithmetic_sequence_count,21.0,2.952381,0.238095,0.238095,0.0
explicit_arithmetic,196.0,0.658163,0.045918,0.045918,0.0
gcd,33.0,0.060606,0.000000,0.000000,0.0
not_rule,3083.0,1.221537,0.088550,0.103146,0.0
rectangle_perimeter,3.0,0.000000,0.000000,0.000000,0.0
square_area,8.0,0.000000,0.000000,0.000000,0.0


In [13]:
# 13. Test inference -> test_predictions.json đúng format
if RUN_TEST_INFERENCE and raw_test_records:
    test_outputs_debug = predict_records_pipeline(raw_test_records, artifacts.get("final_answer_only_type"), True, True, True, True, retrieval_thresholds, name="full_pipeline_test")
    test_public = public_output(test_outputs_debug)
    save_json(test_public, TEST_OUTPUT_PATH)
    print("Saved", TEST_OUTPUT_PATH, "records:", len(test_public))
    print(json.dumps(test_public[:2], ensure_ascii=False, indent=2)[:1600])
else:
    print("Không có test.json hoặc RUN_TEST_INFERENCE=False, bỏ qua test_predictions.json.")

Không có test.json hoặc RUN_TEST_INFERENCE=False, bỏ qua test_predictions.json.


In [14]:
# 14. Final report markdown
chosen_reason = "Chọn final pipeline theo validation score của ablation nếu có; pipeline dùng target mode đã chọn, retrieval top-k vote, rule allowlist và greedy+sampling voting để giảm lỗi decode đơn lẻ."
lines = [
    "# Final report - Hybrid GPT-2 Math Word Problems",
    "",
    "## Data preprocessing",
    f"- Train file: `{TRAIN_FILE}`",
    "- `valid.json` chỉ load tham khảo vì lỗi; validation sinh từ `train.json` bằng grouped stratified split.",
    "- Input chỉ dùng `query_vi`; gold target chỉ dùng `response_vi` và `extract_final_answer(response_vi)`.",
    f"- Train records: {len(train_records)}; valid records: {len(valid_records)}; gold extract failed: {len(extract_fail)}.",
    "",
    "## Target format",
    f"- Final selected target mode: `{FINAL_SELECTED_TARGET_MODE}`; selection: `{json.dumps(FINAL_SELECTION_INFO, ensure_ascii=False)}`.",
    "- Public output vẫn format `Lời giải: ...\\nĐáp án là: <number>`.",
    "",
    "## LoRA config",
    f"- PEFT available: {PEFT_OK}",
    f"- Config: `{json.dumps(LORA_CONFIG, ensure_ascii=False)}`",
    f"- Training args: `{json.dumps(FINAL_TRAINING, ensure_ascii=False)}`",
    "",
    "## Prompt template theo type",
    "```text",
    "Nhiệm vụ: {type_instruction}",
    "Bài toán: {query_vi}",
    "Đáp án:",
    "```",
    "",
    "## Decoding/voting",
    f"- Generation: `{json.dumps(GENERATION, ensure_ascii=False)}`",
    "- Majority vote trên đáp án numeric đã normalize; greedy/beam candidate được weight cao hơn sampling, nếu không extract được thì retry greedy ngắn.",
    "",
    "## Rule/retrieval fallback",
    f"- Rule solver deterministic chỉ override nếu confidence cao và reason nằm trong allowlist: `{json.dumps(sorted(RULE_REASON_ALLOWLIST), ensure_ascii=False)}`.",
    "- Retrieval dùng TF-IDF char_wb 3-5 gram trên train query, top-k vote trong các neighbor rất gần, threshold tune theo type trên valid.",
    f"- Retrieval thresholds: `{json.dumps(retrieval_thresholds, ensure_ascii=False)}`",
    "",
    "## Validation score",
    f"- Overall score_10: {valid_report_payload['final_valid_summary']['score_10']:.4f}",
    f"- Percent of max: {valid_report_payload['final_valid_summary']['score_pct_of_max']:.4f}",
    "",
    "### Score by type",
]
for typ, vals in valid_report_payload["final_valid_summary"]["by_type"].items():
    lines.append(f"- {typ}: score_10={vals['score_10']:.4f}, pass_1pct={vals['pass_1pct_rate']:.4f}, n={vals['num_samples']}")
lines += ["", "## Lý do chọn final configuration", chosen_reason]
FINAL_REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("Saved", FINAL_REPORT_PATH)
print("\n".join(lines[:80]))

Saved /kaggle/working/final_report.md
# Final report - Hybrid GPT-2 Math Word Problems

## Data preprocessing
- Train file: `/kaggle/input/datasets/phamanhtuanas/gpt2-math/keep_asy_if_short/train.json`
- `valid.json` chỉ load tham khảo vì lỗi; validation sinh từ `train.json` bằng grouped stratified split.
- Input chỉ dùng `query_vi`; gold target chỉ dùng `response_vi` và `extract_final_answer(response_vi)`.
- Train records: 85543; valid records: 3344; gold extract failed: 20.

## Target format
- Final selected target mode: `answer_only`; selection: `{"selection": "best_ablation_valid_score", "selected_report": "LoRA + answer-only + prompt + voting", "score_10": 1.2125}`.
- Public output vẫn format `Lời giải: ...\nĐáp án là: <number>`.

## LoRA config
- PEFT available: True
- Config: `{"r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "bias": "none", "task_type": "CAUSAL_LM", "target_modules": ["c_attn", "c_proj", "c_fc"], "fan_in_fan_out": true}`
- Training args: `{"num_train_epochs": 4